In [1]:
# Cell 0: Configuration and Imports

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
from xgboost import XGBClassifier, XGBRegressor
import joblib

# --- Configuration ---
# Replace the command-line arguments with variables
ATHLETES_CSV = 'athletes_events.csv'
WORLD_BANK_CSV = 'world_bank_data.csv'
START_YEAR = 1992  # Per paper, data from 1988 onwards was used. Adjust as needed. 
PROBABILITY_THRESHOLD = 0.5 # Threshold for classifying a country as a medal winner

In [2]:
# Cell 1: Load Datasets

if not os.path.exists(ATHLETES_CSV):
    raise FileNotFoundError(f"{ATHLETES_CSV} not found.")
if not os.path.exists(WORLD_BANK_CSV):
    raise FileNotFoundError(f"{WORLD_BANK_CSV} not found.")

athletes = pd.read_csv(ATHLETES_CSV)
world = pd.read_csv(WORLD_BANK_CSV)

print("Loaded datasets:")
print(f" - athletes: {athletes.shape}")
print(f" - world: {world.shape}")

Loaded datasets:
 - athletes: (271116, 15)
 - world: (422136, 64)


In [3]:
# Cell 2: Process Athlete Data: Aggregate Medal Counts

ath = athletes.copy()
# 1 if a medal was won, 0 otherwise
ath['medal_binary'] = ath['Medal'].notna().astype(int)

# Group by country and year to get total medals and number of athletes
medals = ath.groupby(['NOC', 'Year'], as_index=False).agg(
    medals_total=('medal_binary', 'sum'),
    num_athletes=('ID', 'nunique')
)
medals.rename(columns={'NOC': 'country_code', 'Year': 'year'}, inplace=True)

print("Aggregated medal counts:")
print(medals.head())

Aggregated medal counts:
  country_code  year  medals_total  num_athletes
0          AFG  1936             0            15
1          AFG  1948             0            25
2          AFG  1956             0            12
3          AFG  1960             0            12
4          AFG  1964             0             8


In [4]:
# Cell 3: Process World Bank Data: Reshape to a Usable Format

wb = world.copy()
id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
value_vars = [c for c in wb.columns if c not in id_vars]

# Melt to long format
wb_long = wb.melt(id_vars=id_vars, value_vars=value_vars, var_name='year', value_name='value')
wb_long['year'] = wb_long['year'].astype(str).str.strip()
wb_long = wb_long[wb_long['year'].str.isdigit()]
wb_long['year'] = wb_long['year'].astype(int)

# Pivot indicators into columns
wb_pivot = wb_long.pivot_table(index=['Country Code', 'year'], columns='Indicator Code', values='value', aggfunc='first').reset_index()
wb_pivot.columns.name = None

print("Reshaped World Bank data:")
print(wb_pivot.head())

Reshaped World Bank data:
  Country Code  year  AG.AGR.TRAC.NO  AG.CON.FERT.PT.ZS  AG.CON.FERT.ZS  \
0          ABW  1960             NaN                NaN             NaN   
1          ABW  1961             NaN                NaN             NaN   
2          ABW  1962             NaN                NaN             NaN   
3          ABW  1963             NaN                NaN             NaN   
4          ABW  1964             NaN                NaN             NaN   

   AG.LND.AGRI.K2  AG.LND.AGRI.ZS  AG.LND.ARBL.HA  AG.LND.ARBL.HA.PC  \
0             NaN             NaN             NaN                NaN   
1            20.0       11.111111          2000.0           0.036076   
2            20.0       11.111111          2000.0           0.035571   
3            20.0       11.111111          2000.0           0.035276   
4            20.0       11.111111          2000.0           0.035068   

   AG.LND.ARBL.ZS  ...  per_sa_allsa.cov_q4_tot  per_sa_allsa.cov_q5_tot  \
0             

In [5]:
# Cell 4: Merge Athlete and World Bank Data

merged = medals.merge(wb_pivot, left_on=['country_code', 'year'], right_on=['Country Code', 'year'], how='left')
if 'Country Code' in merged.columns:
    merged.drop(columns=['Country Code'], inplace=True)

# Remove any rows where the target (medals_total) is missing
merged = merged[~merged['medals_total'].isna()].copy()

print("Merged dataset shape:", merged.shape)
print(merged.head())

Merged dataset shape: (3305, 1598)
  country_code  year  medals_total  num_athletes  AG.AGR.TRAC.NO  \
0          AFG  1936             0            15             NaN   
1          AFG  1948             0            25             NaN   
2          AFG  1956             0            12             NaN   
3          AFG  1960             0            12             NaN   
4          AFG  1964             0             8           200.0   

   AG.CON.FERT.PT.ZS  AG.CON.FERT.ZS  AG.LND.AGRI.K2  AG.LND.AGRI.ZS  \
0                NaN             NaN             NaN             NaN   
1                NaN             NaN             NaN             NaN   
2                NaN             NaN             NaN             NaN   
3                NaN             NaN             NaN             NaN   
4                NaN             NaN        378730.0       58.010906   

   AG.LND.ARBL.HA  ...  per_sa_allsa.cov_q4_tot  per_sa_allsa.cov_q5_tot  \
0             NaN  ...                      NaN

In [6]:
# Cell 5: Handle Missing Data and Filter by Year

# Impute missing values with the median for numeric features
exclude = ['country_code', 'year', 'medals_total']
num_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in exclude]

for c in feature_cols:
    median_val = merged[c].median()
    merged[c] = merged[c].fillna(median_val)
merged.fillna(0, inplace=True) # Final fallback for any remaining NaNs

# Filter the dataframe to start from the specified year
merged = merged[merged['year'] >= START_YEAR].reset_index(drop=True)
print(f"Shape after filtering for year >= {START_YEAR}: {merged.shape}")

Shape after filtering for year >= 1992: (1850, 1598)


In [7]:
# Cell 6: Feature Engineering

df = merged.copy()

# Medals per athlete
df['medals_per_athlete'] = df['medals_total'] / df['num_athletes'].replace({0: np.nan})
df['medals_per_athlete'] = df['medals_per_athlete'].fillna(0)

# GDP per capita (heuristically finding GDP and population columns)
possible_gdp = [c for c in df.columns if 'NY.GDP' in str(c) or 'GDP' in str(c)]
possible_pop = [c for c in df.columns if 'SP.POP' in str(c) or 'population' in str(c).lower()]

if possible_gdp and possible_pop:
    gdp_col, pop_col = possible_gdp[0], possible_pop[0]
    df['gdp_per_capita'] = df[gdp_col] / df[pop_col].replace({0: np.nan})
    df['gdp_per_capita'] = df['gdp_per_capita'].fillna(0)
else:
    df['gdp_per_capita'] = 0

# Relative medal share (to account for varying number of medals per Games)
year_totals = df.groupby('year')['medals_total'].sum().rename('total_medals_year')
df = df.merge(year_totals, left_on='year', right_index=True)
df['relative_medal_share'] = df['medals_total'] / df['total_medals_year'].replace({0: np.nan})
df['relative_medal_share'] = df['relative_medal_share'].fillna(0)

# Binary label for Phase 1 classification
df['won_any_medal'] = (df['medals_total'] > 0).astype(int)

# Define the final feature set for modeling
FEATURES = ['num_athletes', 'medals_per_athlete', 'gdp_per_capita', 'relative_medal_share']
FEATURES = [f for f in FEATURES if f in df.columns]

print("Final features used for modeling:", FEATURES)
print(df[FEATURES + ['won_any_medal', 'medals_total']].head())

Final features used for modeling: ['num_athletes', 'medals_per_athlete', 'gdp_per_capita', 'relative_medal_share']
   num_athletes  medals_per_athlete  gdp_per_capita  relative_medal_share  \
0             2            0.000000        0.320763              0.000000   
1             5            0.000000        0.326088              0.000000   
2             4            0.250000        0.338641              0.000488   
3             6            0.166667        0.369692              0.000515   
4             3            0.000000        0.418478              0.000000   

   won_any_medal  medals_total  
0              0             0  
1              0             0  
2              1             1  
3              1             1  
4              0             0  


In [8]:
# Cell 7: Split Data into Tuning, Validation, and Test Sets

# Year splits based on Table 2 from the paper
tuning_years = list(range(1988, 2009, 4)) # 1988-2008 [cite: 84]
validation_years = [2012] # [cite: 84]
test_years = [2016] # [cite: 84]

train_tune_df = df[df['year'].isin(tuning_years)].copy()
val_df = df[df['year'].isin(validation_years)].copy()
test_df = df[df['year'].isin(test_years)].copy()

print(f"Tuning set shape: {train_tune_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

Tuning set shape: (972, 1603)
Validation set shape: (205, 1603)
Test set shape: (207, 1603)


In [9]:
# Cell 8: Phase 1: Classification

# Prepare data and scale features
X_tune = train_tune_df[FEATURES].fillna(0).values
y_tune = train_tune_df['won_any_medal'].values
X_val = val_df[FEATURES].fillna(0).values
y_val = val_df['won_any_medal'].values

scaler = StandardScaler().fit(X_tune)
X_tune_s = scaler.transform(X_tune)
X_val_s = scaler.transform(X_val)

# Candidate classifiers
classifiers = {
    'logreg': LogisticRegression(max_iter=2000),
    'rf': RandomForestClassifier(n_estimators=200, random_state=42),
    'xgb': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# Train and evaluate classifiers on the validation set
results = {}
for name, clf in classifiers.items():
    clf.fit(X_tune_s, y_tune)
    probs = clf.predict_proba(X_val_s)[:, 1]
    auc = roc_auc_score(y_val, probs) if len(np.unique(y_val)) > 1 else float('nan')
    results[name] = auc

print("Validation AUCs (classifiers):", results)

# Select the best classifier and retrain it on the full tuning set
best_clf_name = max(results, key=lambda k: (results[k] if not np.isnan(results[k]) else -1))
best_clf = classifiers[best_clf_name]
best_clf.fit(X_tune_s, y_tune)
print(f"\nChosen classifier: {best_clf_name}")

# Save the trained scaler and classifier
joblib.dump(scaler, 'scaler_phase1.joblib')
joblib.dump(best_clf, 'best_classifier.joblib')

Validation AUCs (classifiers): {'logreg': 1.0, 'rf': 1.0, 'xgb': 1.0}

Chosen classifier: logreg


['best_classifier.joblib']

In [10]:
# Cell 9: Phase 2: Regression

# Use the best classifier to identify positive examples for training the regressor
X_tune_all_s = scaler.transform(train_tune_df[FEATURES].fillna(0).values)
probs_tune = best_clf.predict_proba(X_tune_all_s)[:, 1]
positive_mask = probs_tune >= PROBABILITY_THRESHOLD
reg_train = train_tune_df.loc[positive_mask].copy()

print(f"Regression training set (predicted positives) shape: {reg_train.shape}")

if reg_train.empty:
    print("Warning: No positive examples found for regression training. Try a lower threshold.")
    best_reg = None
else:
    X_reg = reg_train[FEATURES].fillna(0).values
    y_reg = reg_train['medals_total'].values

    # Filter validation set similarly to get a regression validation set
    X_val_all_s = scaler.transform(val_df[FEATURES].fillna(0).values)
    probs_val = best_clf.predict_proba(X_val_all_s)[:, 1]
    val_pos_mask = probs_val >= PROBABILITY_THRESHOLD
    reg_val = val_df.loc[val_pos_mask].copy()
    print(f"Regression validation set (predicted positives) shape: {reg_val.shape}")
    
    X_reg_val = reg_val[FEATURES].fillna(0).values if not reg_val.empty else np.empty((0, len(FEATURES)))
    y_reg_val = reg_val['medals_total'].values if not reg_val.empty else np.array([])
    
    # Candidate regressors
    regressors = {
        'linreg': LinearRegression(),
        'rf': RandomForestRegressor(n_estimators=200, random_state=42),
        'xgb': XGBRegressor(random_state=42)
    }

    # Train and evaluate regressors
    reg_results = {}
    for name, reg in regressors.items():
        reg.fit(X_reg, y_reg)
        preds = reg.predict(X_reg_val) if X_reg_val.shape[0] > 0 else np.array([])
        mse = mean_squared_error(y_reg_val, preds) if preds.size > 0 else float('inf')
        reg_results[name] = mse

    print("\nRegression validation MSEs:", reg_results)
    
    # Select the best regressor and retrain
    best_reg_name = min(reg_results, key=lambda k: (reg_results[k] if not np.isnan(reg_results[k]) else float('inf')))
    best_reg = regressors[best_reg_name]
    best_reg.fit(X_reg, y_reg)
    print(f"\nChosen regressor: {best_reg_name}")
    joblib.dump(best_reg, 'best_regressor.joblib')

Regression training set (predicted positives) shape: (375, 1603)
Regression validation set (predicted positives) shape: (84, 1603)

Regression validation MSEs: {'linreg': 1.6751424924882297, 'rf': 10.330296428571414, 'xgb': 4.8101654052734375}

Chosen regressor: linreg


In [11]:
# Cell 10: Hyperparameter Tuning (Optional)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Example: Randomized search for Random Forest Classifier
print("\nTuning Random Forest Classifier...")
rf_param_dist = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}
r_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_param_dist,
    n_iter=6, cv=kf, scoring='roc_auc', random_state=42
)
r_search.fit(X_tune_s, y_tune)
print("RF tuned best params:", r_search.best_params_)
joblib.dump(r_search.best_estimator_, 'rf_tuned.joblib')

# Example: Grid search for Logistic Regression
print("\nTuning Logistic Regression Classifier...")
lg_param_grid = {'C': [0.01, 0.1, 1, 10]}
lg_grid = GridSearchCV(LogisticRegression(max_iter=2000), lg_param_grid, cv=kf, scoring='roc_auc')
lg_grid.fit(X_tune_s, y_tune)
print("LogReg tuned best params:", lg_grid.best_params_)
joblib.dump(lg_grid.best_estimator_, 'logreg_tuned.joblib')


Tuning Random Forest Classifier...
RF tuned best params: {'n_estimators': 400, 'min_samples_split': 10, 'max_depth': 20}

Tuning Logistic Regression Classifier...
LogReg tuned best params: {'C': 10}


['logreg_tuned.joblib']

In [13]:
# Cell 11: Final Evaluation on the Test Set

# Load the best (untuned) models for final evaluation
scaler = joblib.load('scaler_phase1.joblib')
best_clf = joblib.load('best_classifier.joblib')
if os.path.exists('best_regressor.joblib'):
    best_reg = joblib.load('best_regressor.joblib')
else:
    best_reg = None

# Prepare test data
X_test = test_df[FEATURES].fillna(0).values
y_test_class = test_df['won_any_medal'].values
X_test_s = scaler.transform(X_test)

# Phase 1 evaluation
probs_test = best_clf.predict_proba(X_test_s)[:, 1]
auc_test = roc_auc_score(y_test_class, probs_test)
print(f"\n--- Final Test Set Evaluation ---")
print(f"Test ROC AUC (Classification): {auc_test:.4f}")

# Phase 2 evaluation
pred_pos_mask_test = probs_test >= PROBABILITY_THRESHOLD
reg_test_df = test_df.loc[pred_pos_mask_test].copy()
print(f"Predicted positives in test set: {reg_test_df.shape[0]}")

if best_reg is not None and not reg_test_df.empty:
    X_reg_test = reg_test_df[FEATURES].fillna(0).values
    y_reg_test = reg_test_df['medals_total'].values
    
    preds_reg = best_reg.predict(X_reg_test)
    
    mse_test = mean_squared_error(y_reg_test, preds_reg)
    r2 = r2_score(y_reg_test, preds_reg)
    
    print(f"Regression Test MSE: {mse_test:.4f}")
    print(f"Regression Test R^2: {r2:.4f}")
    
    # Add predictions to the dataframe for inspection
    reg_test_df['predicted_medals'] = preds_reg
    print("\nSample of test predictions:")
    print(reg_test_df[['country_code', 'medals_total', 'predicted_medals']].sort_values('medals_total', ascending=False).head(10))
    
else:
    print("No regression evaluation performed (no best regressor or no predicted positives).")


--- Final Test Set Evaluation ---
Test ROC AUC (Classification): 1.0000
Predicted positives in test set: 83
Regression Test MSE: 0.6240
Regression Test R^2: 0.9996

Sample of test predictions:
     country_code  medals_total  predicted_medals
1782          USA           264        259.620606
668           GER           159        156.256571
630           GBR           145        142.673078
1454          RUS           115        113.187566
365           CHN           113        111.174218
598           FRA            96         94.461998
108           AUS            82         80.644779
867           ITA            72         71.065167
316           CAN            69         67.967720
907           JPN            64         63.179259


In [ ]:
# Cell 12: Save the Final Pipeline

pipeline_objects = {
    'scaler': scaler,
    'classifier': best_clf,
    'regressor': best_reg
}

joblib.dump(pipeline_objects, 'final_pipeline.joblib')
print("\nSaved final pipeline components to 'final_pipeline.joblib'")


Saved final pipeline components to 'final_pipeline.joblib'
